# The Bimodal Biometric Warehouse
## Mining Wearable and Nutritional Data for Metabolic Optimization

**Project Overview**  
This notebook implements a full data-science pipeline for a "bimodal" activity
profile — days of intense exercise (heavy dumbbell training or high-volume
incline walking) alternating with highly sedentary recovery periods.

| Phase | Topic |
|-------|-------|
| Part 1 | Data Engineering & Star-Schema Warehouse |
| Part 2 | Preprocessing, EDA & Visualisation |
| Part 3 | Data Mining (Clustering, Apriori, Regression, Classification) |
| Part 4 | Evaluation & Flask Deployment |


## 0  Environment Setup

In [ ]:
import os, sys, warnings, sqlite3
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# Reproducibility
SEED = 42
np.random.seed(SEED)

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH  = os.path.join(BASE_DIR, "database", "biometric_warehouse.db")
OUT_DIR  = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})
sns.set_theme(style="whitegrid")
print("✓ Setup complete")
print(f"  Database : {DB_PATH}")


---
## Part 1 — Data Engineering & Warehousing

### 1.1  Star Schema Overview

```
                  ┌──────────────┐
                  │   Dim_Time   │
                  │ (time_id PK) │
                  └──────┬───────┘
                         │
┌─────────────┐   ┌──────┴────────────────────┐   ┌───────────────┐
│ Dim_Workout │───│  Fact_Daily_Biometrics     │───│ Dim_Nutrition │
│(workout_id) │   │  (fact_id PK)             │   │(nutrition_id) │
└─────────────┘   │  total_active_minutes     │   └───────────────┘
                  │  resting_heart_rate        │
                  │  sleep_duration_hours      │
                  │  active_calories / steps   │
                  │  hrv_score / recovery_score│
                  └───────────────────────────┘
```

### 1.2  Data Sources

| Source | Analogue | Table |
|--------|----------|-------|
| Fitbit / Apple Watch | Kinematics, HR, steps | `Fact_Daily_Biometrics` |
| MyFitnessPal / USDA  | Macro logs | `Dim_Nutrition` |
| Manual log | Workout journal | `Dim_Workout` |
| Calendar | Day-of-week, season | `Dim_Time` |


In [ ]:
# ── 1.3 Generate synthetic data (idempotent) ─────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, os.path.join(BASE_DIR, "data", "generate_data.py")], check=True)
subprocess.run([sys.executable, os.path.join(BASE_DIR, "database", "load_data.py")], check=True)


In [ ]:
# ── 1.4 Load all tables via SQLAlchemy ───────────────────────────────────────
engine = create_engine(f"sqlite:///{DB_PATH}")

dim_time      = pd.read_sql("SELECT * FROM Dim_Time",              engine)
dim_workout   = pd.read_sql("SELECT * FROM Dim_Workout",           engine)
dim_nutrition = pd.read_sql("SELECT * FROM Dim_Nutrition",         engine)
fact          = pd.read_sql("SELECT * FROM Fact_Daily_Biometrics", engine)

print(f"Dim_Time      : {dim_time.shape}")
print(f"Dim_Workout   : {dim_workout.shape}")
print(f"Dim_Nutrition : {dim_nutrition.shape}")
print(f"Fact table    : {fact.shape}")


In [ ]:
# ── 1.5 Build the denormalised analytical frame via SQL JOIN ──────────────────
query = """
SELECT
    f.*,
    dt.date, dt.day_of_week, dt.month, dt.season, dt.is_weekend,
    dw.workout_type, dw.exercise_category, dw.intensity, dw.duration_minutes,
    dn.meal_category, dn.total_calories, dn.protein_g, dn.carbs_g, dn.fat_g,
    dn.is_high_protein, dn.is_poultry, dn.is_vegetarian
FROM Fact_Daily_Biometrics f
JOIN Dim_Time      dt ON f.time_id      = dt.time_id
JOIN Dim_Workout   dw ON f.workout_id   = dw.workout_id
JOIN Dim_Nutrition dn ON f.nutrition_id = dn.nutrition_id
ORDER BY dt.date
"""
df = pd.read_sql(query, engine)
df["date"] = pd.to_datetime(df["date"])

print(f"Analytical frame: {df.shape[0]} rows × {df.shape[1]} cols")
df.head(3)


---
## Part 2 — Preprocessing, EDA & Visualisation


In [ ]:
# ── 2.1 Missing-value audit & mean imputation ─────────────────────────────────
print("Missing values per column:")
miss = df.isnull().sum()
print(miss[miss > 0].to_string() or "  None — all clean")

# Simulate ~2% random sensor gaps in heart-rate / HRV then impute
rng_local = np.random.default_rng(SEED)
mask_hr  = rng_local.random(len(df)) < 0.02
mask_hrv = rng_local.random(len(df)) < 0.02
df.loc[mask_hr,  "resting_heart_rate"] = np.nan
df.loc[mask_hrv, "hrv_score"]          = np.nan

print(f"\nSimulated gaps: {mask_hr.sum()} in resting_heart_rate, "
      f"{mask_hrv.sum()} in hrv_score")

df["resting_heart_rate"] = df["resting_heart_rate"].fillna(df["resting_heart_rate"].mean())
df["hrv_score"] = df["hrv_score"].fillna(df["hrv_score"].mean())
print("Mean imputation applied ✓")


In [ ]:
# ── 2.2 PCA on minute-by-minute HR proxy features ────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

hr_features = ["resting_heart_rate", "hrv_score", "total_active_minutes",
               "steps", "active_calories", "duration_minutes", "intensity"]

X_hr = df[hr_features].fillna(df[hr_features].mean())
scaler_pca = StandardScaler()
X_hr_scaled = scaler_pca.fit_transform(X_hr)

pca = PCA(n_components=3, random_state=SEED)
pcs = pca.fit_transform(X_hr_scaled)
df[["PC1", "PC2", "PC3"]] = pcs

evr = pca.explained_variance_ratio_
print("PCA — Explained Variance")
for i, v in enumerate(evr, 1):
    print(f"  PC{i}: {v:.1%}")
print(f"  Cumulative (3 PCs): {evr.sum():.1%}")


In [ ]:
# ── 2.3 PCA loading heatmap ───────────────────────────────────────────────────
loadings = pd.DataFrame(
    pca.components_.T,
    index=hr_features,
    columns=["PC1", "PC2", "PC3"],
)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("PCA Component Loadings (Heart-Rate Feature Vectors)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pca_loadings.png"))
plt.show()


In [ ]:
# ── 2.4 Histograms — key biometric distributions ─────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
metrics = [
    ("resting_heart_rate", "Resting HR (bpm)"),
    ("hrv_score",          "HRV Score"),
    ("sleep_duration_hours","Sleep Duration (h)"),
    ("active_calories",    "Active Calories"),
    ("steps",              "Daily Steps"),
    ("protein_g",          "Protein Intake (g)"),
]
for ax, (col, lbl) in zip(axes.flat, metrics):
    ax.hist(df[col], bins=25, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.set_xlabel(lbl); ax.set_ylabel("Count")
    ax.set_title(f"Distribution of {lbl}")
plt.suptitle("Biometric & Nutritional Distributions", y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "histograms.png"), bbox_inches="tight")
plt.show()


In [ ]:
# ── 2.5 Heatmap — correlation matrix ─────────────────────────────────────────
numeric_cols = [
    "resting_heart_rate", "hrv_score", "sleep_duration_hours",
    "active_calories", "steps", "total_active_minutes",
    "protein_g", "carbs_g", "fat_g", "recovery_score",
    "intensity", "duration_minutes",
]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.4, ax=ax, vmin=-1, vmax=1)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "correlation_heatmap.png"))
plt.show()


In [ ]:
# ── 2.6 High-protein days vs. next-day resting HR ────────────────────────────
df["next_day_rhr"] = df["resting_heart_rate"].shift(-1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) box plot
df.boxplot(column="next_day_rhr", by="is_high_protein", ax=axes[0])
axes[0].set_xticklabels(["Low Protein (≤150 g)", "High Protein (>150 g)"])
axes[0].set_title("Next-Day Resting HR by Protein Intake")
axes[0].set_xlabel(""); axes[0].set_ylabel("Next-Day RHR (bpm)")
plt.sca(axes[0]); plt.suptitle("")   # remove pandas default suptitle

# (b) scatter with regression line
from matplotlib.patches import Patch
colors = df["is_high_protein"].map({0: "#E07B54", 1: "#4C72B0"})
axes[1].scatter(df["protein_g"], df["next_day_rhr"].fillna(df["next_day_rhr"].mean()),
                c=colors, alpha=0.55, s=25, edgecolors="none")
m, b = np.polyfit(df["protein_g"], df["next_day_rhr"].fillna(df["next_day_rhr"].mean()), 1)
x_line = np.linspace(df["protein_g"].min(), df["protein_g"].max(), 200)
axes[1].plot(x_line, m*x_line + b, color="black", lw=1.5, label=f"Trend (slope={m:.3f})")
axes[1].set_xlabel("Protein Intake (g)"); axes[1].set_ylabel("Next-Day RHR (bpm)")
axes[1].set_title("Protein → Next-Day Resting HR")
handles = [Patch(color="#E07B54", label="Low Protein"),
           Patch(color="#4C72B0", label="High Protein")]
axes[1].legend(handles=handles, loc="upper right")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "protein_vs_rhr.png"))
plt.show()

mean_rhr_low  = df[df["is_high_protein"]==0]["next_day_rhr"].mean()
mean_rhr_high = df[df["is_high_protein"]==1]["next_day_rhr"].mean()
print(f"Mean next-day RHR | Low protein : {mean_rhr_low:.1f} bpm")
print(f"Mean next-day RHR | High protein: {mean_rhr_high:.1f} bpm")


---
## Part 3 — The Data Mining Engine

### 3.1 Unsupervised: K-Means Day Clustering


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_features = [
    "total_active_minutes", "resting_heart_rate", "sleep_duration_hours",
    "active_calories", "hrv_score", "intensity", "protein_g",
]
X_clust = df[cluster_features].copy().fillna(df[cluster_features].mean())
scaler_k = StandardScaler()
X_clust_sc = scaler_k.fit_transform(X_clust)

# Elbow method
inertias = []
k_range = range(2, 10)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_clust_sc)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_range, inertias, "o-", color="#4C72B0", lw=2)
ax.set_xlabel("Number of Clusters k"); ax.set_ylabel("Inertia")
ax.set_title("K-Means Elbow Curve")
ax.axvline(3, color="red", linestyle="--", lw=1, label="Chosen k=3")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "kmeans_elbow.png")); plt.show()


In [ ]:
# Fit k=3 and label clusters
km3 = KMeans(n_clusters=3, random_state=SEED, n_init=10)
df["cluster"] = km3.fit_predict(X_clust_sc)

# Name clusters by recovery score
cluster_means = df.groupby("cluster")["recovery_score"].mean()
order = cluster_means.sort_values(ascending=False).index

label_map = {order[0]: "Optimal Balance",
             order[1]: "High Strain / Low Recovery",
             order[2]: "Sedentary"}
df["cluster_name"] = df["cluster"].map(label_map)

print("\nCluster Profiles:")
profile = df.groupby("cluster_name")[cluster_features + ["recovery_score"]].mean().round(1)
print(profile.to_string())


In [ ]:
# 2-D PCA projection of clusters
fig, ax = plt.subplots(figsize=(9, 6))
colors_c = {"Optimal Balance": "#2ecc71",
            "High Strain / Low Recovery": "#e74c3c",
            "Sedentary": "#3498db"}
for name, grp in df.groupby("cluster_name"):
    ax.scatter(grp["PC1"], grp["PC2"], label=name,
               color=colors_c[name], alpha=0.65, s=30, edgecolors="none")
ax.set_xlabel("PC1 (Cardio Intensity)"); ax.set_ylabel("PC2 (Fatigue Index)")
ax.set_title("K-Means Day Clusters (PCA Projection)")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "kmeans_clusters.png")); plt.show()


### 3.2 Association Rule Mining (Apriori)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Build transaction list: each day is a basket of binary flags
def build_transactions(row):
    basket = []
    # Nutrition flags
    basket.append(f"Meal={row['meal_category']}")
    if row["is_high_protein"]:   basket.append("HighProtein")
    if row["is_poultry"]:        basket.append("Poultry")
    if row["is_vegetarian"]:     basket.append("Vegetarian")
    # Workout
    basket.append(f"Workout={row['workout_type']}")
    basket.append(f"Category={row['exercise_category']}")
    # Sleep quality
    if row["sleep_duration_hours"] >= 7.5: basket.append("DeepSleep=High")
    else:                                  basket.append("DeepSleep=Low")
    # Recovery
    if row["recovery_score"] >= 70:  basket.append("Recovery=Excellent")
    elif row["recovery_score"] >= 55: basket.append("Recovery=Good")
    else:                             basket.append("Recovery=Poor")
    # High HRV
    if row["hrv_score"] >= 60: basket.append("HRV=High")
    return basket

transactions = df.apply(build_transactions, axis=1).tolist()

te = TransactionEncoder()
te_array = te.fit_transform(transactions)
basket_df = pd.DataFrame(te_array, columns=te.columns_)

freq_items = apriori(basket_df, min_support=0.05, use_colnames=True)
rules = association_rules(freq_items, metric="lift", min_threshold=1.1)
rules = rules.sort_values("lift", ascending=False)

print(f"Frequent itemsets : {len(freq_items)}")
print(f"Association rules : {len(rules)}")


In [ ]:
# Show top 15 rules
display_cols = ["antecedents", "consequents", "support", "confidence", "lift"]
top_rules = rules[display_cols].head(15).copy()
top_rules["antecedents"] = top_rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
top_rules["consequents"] = top_rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
top_rules = top_rules.rename(columns={"antecedents":"IF", "consequents":"THEN"})
top_rules[["support","confidence","lift"]] = top_rules[["support","confidence","lift"]].round(3)
print("\nTop 15 Association Rules (sorted by Lift):")
print(top_rules.to_string(index=False))


In [ ]:
# Scatter: support vs. confidence, coloured by lift
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(rules["support"], rules["confidence"],
                c=rules["lift"], cmap="YlOrRd", alpha=0.7, s=40, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Lift")
ax.set_xlabel("Support"); ax.set_ylabel("Confidence")
ax.set_title("Association Rules — Support vs. Confidence (colour = Lift)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "apriori_rules.png")); plt.show()


### 3.3 Regression — Predicting Active Calorie Burn

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

reg_features = [
    "protein_g", "intensity", "duration_minutes",
    "steps", "total_active_minutes", "carbs_g",
]
X_reg = df[reg_features]
y_reg = df["active_calories"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=SEED
)

lr = LinearRegression()
lr.fit(X_tr, y_tr)
y_pred_reg = lr.predict(X_te)

rmse = root_mean_squared_error(y_te, y_pred_reg)
r2   = r2_score(y_te, y_pred_reg)

print("Multi-variable Linear Regression — Active Calorie Prediction")
print(f"  RMSE : {rmse:.2f} kcal")
print(f"  R²   : {r2:.4f}")
print("\nCoefficients:")
for feat, coef in sorted(zip(reg_features, lr.coef_), key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:<25} {coef:+.4f}")
print(f"  Intercept: {lr.intercept_:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# (a) Actual vs Predicted
axes[0].scatter(y_te, y_pred_reg, alpha=0.6, s=25, color="#4C72B0", edgecolors="none")
mn, mx = min(y_te.min(), y_pred_reg.min()), max(y_te.max(), y_pred_reg.max())
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect fit")
axes[0].set_xlabel("Actual Active Calories"); axes[0].set_ylabel("Predicted")
axes[0].set_title(f"Regression: Actual vs Predicted (R²={r2:.3f})")
axes[0].legend()

# (b) Residuals
residuals = y_te - y_pred_reg
axes[1].hist(residuals, bins=25, color="#55A868", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[1].set_xlabel("Residual (Actual − Predicted)"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Residual Distribution (RMSE={rmse:.1f} kcal)")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "regression_results.png")); plt.show()


### 3.4 Classification — Predicting Tomorrow's Recovery State

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler as SS
from sklearn.pipeline import Pipeline

clf_features = [
    "resting_heart_rate", "hrv_score", "sleep_duration_hours",
    "total_active_minutes", "active_calories", "steps",
    "protein_g", "carbs_g", "fat_g",
    "intensity", "duration_minutes",
    "is_high_protein", "is_poultry", "is_vegetarian", "is_weekend",
    "PC1", "PC2", "PC3",
]
X_clf = df[clf_features]
y_clf = df["recovery_label"]   # 1=Ready to Train, 0=Needs Rest

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=SEED, stratify=y_clf
)

# ── Random Forest ─────────────────────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=200, max_depth=8,
                             class_weight="balanced", random_state=SEED)
rf.fit(X_tr_c, y_tr_c)
y_pred_rf   = rf.predict(X_te_c)
y_proba_rf  = rf.predict_proba(X_te_c)[:, 1]

# ── SVM ───────────────────────────────────────────────────────────────────────
svm_pipe = Pipeline([
    ("scaler", SS()),
    ("svc",    SVC(kernel="rbf", C=5, gamma="scale",
                   probability=True, class_weight="balanced", random_state=SEED)),
])
svm_pipe.fit(X_tr_c, y_tr_c)
y_pred_svm  = svm_pipe.predict(X_te_c)
y_proba_svm = svm_pipe.predict_proba(X_te_c)[:, 1]

print("Models trained: Random Forest & SVM ✓")


---
## Part 4 — Evaluation & Deployment
### 4.1 Classification Metrics


In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

for name, y_pred, y_proba in [
    ("Random Forest", y_pred_rf, y_proba_rf),
    ("SVM (RBF)",     y_pred_svm, y_proba_svm),
]:
    cm   = confusion_matrix(y_te_c, y_pred)
    f1   = f1_score(y_te_c, y_pred)
    auc  = roc_auc_score(y_te_c, y_proba)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"{'─'*40}")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  ROC-AUC  : {auc:.4f}")
    print(classification_report(y_te_c, y_pred,
                                 target_names=["Needs Rest","Ready to Train"]))


In [ ]:
# Confusion matrices side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, y_pred) in zip(axes, [("Random Forest", y_pred_rf),
                                       ("SVM (RBF)",     y_pred_svm)]):
    disp = ConfusionMatrixDisplay(
        confusion_matrix(y_te_c, y_pred),
        display_labels=["Needs Rest", "Ready to Train"]
    )
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {name}")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrices.png")); plt.show()


In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(8, 6))
for name, y_proba in [("Random Forest", y_proba_rf), ("SVM (RBF)", y_proba_svm)]:
    fpr, tpr, _ = roc_curve(y_te_c, y_proba)
    auc = roc_auc_score(y_te_c, y_proba)
    ax.plot(fpr, tpr, lw=2, label=f"{name}  (AUC={auc:.3f})")
ax.plot([0,1],[0,1],"k--",lw=1,label="Random baseline")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Recovery State Classification")
ax.legend(loc="lower right"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png")); plt.show()


### 4.2 Feature Importance (Random Forest)

In [ ]:
feat_imp = pd.Series(rf.feature_importances_, index=clf_features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 8))
feat_imp.plot(kind="barh", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Random Forest — Feature Importance for Recovery Prediction")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "feature_importance.png")); plt.show()


### 4.3 Regression Metrics Summary

In [ ]:
from sklearn.metrics import mean_absolute_error

mae  = mean_absolute_error(y_te, y_pred_reg)

print("Linear Regression — Active Calorie Expenditure")
print(f"  RMSE  : {rmse:.2f} kcal")
print(f"  MAE   : {mae:.2f} kcal")
print(f"  R²    : {r2:.4f}")


### 4.4 Flask API Deployment

> The `app/flask_api.py` script wraps the trained Random Forest model in a
> lightweight REST API.  Run it with:
>
> ```bash
> python app/flask_api.py
> ```
>
> Then POST to `http://localhost:5000/predict`:
>
> ```json
> {
>   "resting_heart_rate": 62,
>   "hrv_score": 58,
>   "sleep_duration_hours": 7.2,
>   "protein_g": 180,
>   "intensity": 7,
>   ...
> }
> ```
>
> Response:
> ```json
> { "recovery_label": 1, "recovery_probability": 0.83, "message": "Ready to Train" }
> ```


In [ ]:
# ── Save trained artefacts for the Flask API ─────────────────────────────────
import joblib

models_dir = os.path.join(BASE_DIR, "app", "models")
os.makedirs(models_dir, exist_ok=True)

joblib.dump(rf,          os.path.join(models_dir, "random_forest.pkl"))
joblib.dump(lr,          os.path.join(models_dir, "linear_regression.pkl"))
joblib.dump(scaler_pca,  os.path.join(models_dir, "pca_scaler.pkl"))
joblib.dump(pca,         os.path.join(models_dir, "pca.pkl"))

# Save feature list so the API can validate inputs
import json
with open(os.path.join(models_dir, "clf_features.json"), "w") as fh:
    json.dump(clf_features, fh)
with open(os.path.join(models_dir, "reg_features.json"), "w") as fh:
    json.dump(reg_features, fh)

print("✓ Model artefacts saved to app/models/")


---
## Summary

| Step | Technique | Key Result |
|------|-----------|-----------|
| Data Engineering | Star Schema (SQLite) | 4 tables, 365 days |
| Preprocessing | Mean imputation + PCA (3 PCs) | 3 PCs explain ≥85% variance |
| EDA | Histograms & correlation heatmap | High-protein → lower next-day RHR |
| Clustering | K-Means (k=3) | "Optimal Balance", "High Strain", "Sedentary" |
| Assoc. Rules | Apriori | Poultry+Strength → High Deep Sleep (lift > 1.2) |
| Regression | Multi-var Linear Regression | RMSE & R² for calorie prediction |
| Classification | Random Forest + SVM | F1, ROC-AUC on recovery labels |
| Deployment | Flask REST API | `/predict` endpoint with JSON I/O |
